In [ ]:

# Show current directory
import os
curr_dir = os.getcwd()
print(curr_dir)
import numpy as np
from collections import deque

In [ ]:
import sys
parent_dir = os.path.abspath(os.path.join(curr_dir, os.pardir))
# Add the grandparent directory to the system path
# grandparent_dir = os.path.abspath(os.path.join(curr_dir, os.pardir, os.pardir))
sys.path.append(parent_dir)
print(sys.path)

In [ ]:
# unit: timesteps (ms) - The time window after the GT annotation where the network should predict the burst (GT_time, GT_time + PRED_CAUSALITY_WINDOW)
# This is needed to give the network some extra time steps to increase the membrane potential and spike
RIPPLE_DETECTION_OFFSET = [18, 45, 31, 20]
PRED_CAUSALITY_WINDOW = int(5)     # Giving PRED_CAUSALITY_WINDOW ms for the network to update its inner state and spike  
# in timesteps (ms) - Max time from the Insertion Timing to the GT annotation

add_tolerance=True

TOLERANCE= RIPPLE_DETECTION_OFFSET[3] if add_tolerance else 0 # in timesteps (ms) - Tolerance for the prediction window, to account for the fact that the network might not be able to predict the burst exactly at the GT annotation time

MAX_DETECTION_OFFSET = int(RIPPLE_DETECTION_OFFSET[1]) * 1.5 + PRED_CAUSALITY_WINDOW + TOLERANCE # in timesteps (ms)

print(f"PRED_CAUSALITY_WINDOW: {PRED_CAUSALITY_WINDOW}")
print(f"MAX_DETECTION_OFFSET: {MAX_DETECTION_OFFSET} ms")

In [ ]:
downsampled_fs="30000_1000"
data_dir=os.path.join(parent_dir,"extract_Nripples","train_pedro","dataset_up_down",str(downsampled_fs))

In [ ]:
concat_data=np.load(os.path.join(data_dir,"concat_spikes.npy"),allow_pickle=True)
ripples_GT=np.load(os.path.join(data_dir,"concat_ripples.npy"),allow_pickle=True)

In [ ]:
print("Number of UP spikes: ", np.sum(concat_data[:, 0]))
print("Number of DN spikes: ", np.sum(concat_data[:, 1]))

In [ ]:
#### CHOOSE A SEED FOR REPRODUCIBILITY
seeded = True  # Set to True if you want to use a specific seed for reproducibility
if seeded:
    seed = 9
    time_duration= 600 # in seconds, the duration of the test
    window=np.arange(seed*time_duration*1000, (time_duration+seed*time_duration)*1000, 1) # 1000 ms window
    cut_data=concat_data[window]
    ripples_window=[]
    print("Window:", window[0], window[-1])
    for ripple in ripples_GT:
        if ripple[1] >= window[0] and ripple[0] <= window[-1]:
            ripples_window.append(ripple)
    ripples_window=np.array(ripples_window)
    cut_ripples_GT=ripples_window-window[0]  # Normalize the GT ripples to the window start time
else:
    cut_data=concat_data
    cut_ripples_GT=ripples_GT


In [ ]:
# Define the number of total timesteps
total_num_steps = cut_data.shape[0]
num_hfo_events = cut_ripples_GT.shape[0]
num_hfo_timesteps=np.sum(ripples_GT[:, 1] - ripples_GT[:, 0])
print(f"Number of HFO Events: {num_hfo_events}")
print(f"Total number of timesteps: {total_num_steps}")
print("Num of Ripple timesteps:", num_hfo_timesteps)

In [ ]:
# Transform ripples_GT into the time of onset...
ripples_start=cut_ripples_GT[:,0]-TOLERANCE
print("Ripples start: ", ripples_start[:10])

In [ ]:
distance_ripples=ripples_start[1:] - ripples_start[:-1]
average_distance=np.mean(distance_ripples)
median_distance=np.median(distance_ripples)
min_distance=np.min(distance_ripples)
percentile_25=np.percentile(distance_ripples, 10)                            
print(f"Average distance between ripples: {average_distance} ms")
print(f"Median distance between ripples: {median_distance} ms")
print(f"Minimum distance between ripples: {min_distance} ms")
print(f"25th Percentile distance between ripples: {percentile_25} ms")


In [ ]:
refrac_period=100

In [ ]:
## LOAD OUTPUT SPIKES
spikes_dir= os.path.join(curr_dir,"eval","spikes")
prefix="updnb4ds_100_7"
if seeded:
    # If using a specific seed, load the spikes file with the seed in the name
    spikes_file=os.path.join(spikes_dir, f"{prefix}_spikes_seed{seed}.npy")
else:
    # If not using a specific seed, load the spikes file without the seed in the name
    spikes_file=os.path.join(spikes_dir, f"{prefix}_spikes.npy")
out_spikes=np.load(spikes_file, allow_pickle=True)

In [ ]:
from utils.permutation_test import *
# Run the permutation test
actual_score,null_distribution,p_value=live_permutation_test(ripples_start,len(cut_data),out_spikes,MAX_DETECTION_OFFSET,refrac_period=refrac_period)